# Fine-tuning Whisper Large-V3 sur le Wolof
Ce notebook permet d'entraîner Whisper sur des données wolof pour améliorer drastiquement la précision.

**Prérequis** : Activer le GPU dans Colab → Runtime → Change runtime type → T4 GPU

## Sources de données utilisées :
1. Mozilla Common Voice (Wolof)
2. ALFFA Dataset (langues africaines)
3. Vos propres données (audio + transcription)

In [ ]:
# Installation des dépendances
!pip install -q transformers datasets accelerate evaluate jiwer tensorboard
!pip install -q soundfile librosa
!pip install -q huggingface_hub

In [ ]:
# Connexion à Hugging Face (pour sauvegarder le modèle)
from huggingface_hub import notebook_login
notebook_login()

## Étape 1 : Charger les données Mozilla Common Voice (Wolof)

In [ ]:
from datasets import load_dataset, DatasetDict, Audio, concatenate_datasets

# Charger Common Voice Wolof
# Tu dois accepter les conditions sur : https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
common_voice = DatasetDict()

common_voice["train"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "wo",  # Wolof
    split="train+validation",
    trust_remote_code=True,
)

common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "wo",
    split="test",
    trust_remote_code=True,
)

print(f"Train: {len(common_voice['train'])} exemples")
print(f"Test: {len(common_voice['test'])} exemples")

In [ ]:
# Nettoyer les colonnes inutiles
columns_to_remove = [
    "accent", "age", "client_id", "down_votes", "gender",
    "locale", "path", "segment", "up_votes", "variant"
]

common_voice = common_voice.remove_columns(
    [col for col in columns_to_remove if col in common_voice["train"].column_names]
)

print(common_voice["train"][0])

## Étape 2 : Ajouter vos propres données

Pour ajouter votre audio de 2h+ transcrit :
1. Découpez l'audio en segments de 10-30 secondes
2. Créez un CSV avec colonnes : `audio_path`, `sentence`
3. Uploadez sur Google Drive et montez-le

In [ ]:
# Montez Google Drive si vous avez des données perso
from google.colab import drive
drive.mount('/content/drive')

# Structure attendue dans votre Drive :
# /content/drive/MyDrive/wolof_data/
#   ├── audios/
#   │   ├── segment_001.wav
#   │   ├── segment_002.wav
#   │   └── ...
#   └── transcriptions.csv  (colonnes: audio_path, sentence)

import os
import pandas as pd

CUSTOM_DATA_PATH = "/content/drive/MyDrive/wolof_data"

if os.path.exists(CUSTOM_DATA_PATH):
    df = pd.read_csv(f"{CUSTOM_DATA_PATH}/transcriptions.csv")
    print(f"Données perso trouvées: {len(df)} segments")
    
    # Créer un dataset à partir de vos données
    from datasets import Dataset
    
    custom_dataset = Dataset.from_pandas(df)
    custom_dataset = custom_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
    custom_dataset = custom_dataset.rename_column("audio_path", "audio")
    
    # Combiner avec Common Voice
    common_voice["train"] = concatenate_datasets([common_voice["train"], custom_dataset])
    print(f"Dataset combiné: {len(common_voice['train'])} exemples au total")
else:
    print("Pas de données perso trouvées, on continue avec Common Voice uniquement")
    print(f"Créez le dossier: {CUSTOM_DATA_PATH}")

## Étape 3 : Préparer le Feature Extractor et le Tokenizer

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

model_name = "openai/whisper-large-v3"

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
tokenizer = WhisperTokenizer.from_pretrained(model_name, language="wo", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_name, language="wo", task="transcribe")

print("Processor chargé !")

In [ ]:
# Rééchantillonner l'audio à 16kHz
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print("Audio rééchantillonné à 16kHz")

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    
    # Extraire les features audio
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Encoder le texte cible
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    
    return batch

# Appliquer le prétraitement
common_voice = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice.column_names["train"],
    num_proc=1,
)

print("Dataset prétraité !")

## Étape 4 : Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
)

## Étape 5 : Métriques d'évaluation (WER)

In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Étape 6 : Charger le modèle et configurer l'entraînement

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Configurer pour le wolof
model.generation_config.language = "wo"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print(f"Modèle chargé: {model_name}")
print(f"Paramètres: {model.num_parameters() / 1e6:.0f}M")

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-wolof-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=2000,  # Augmentez si vous avez plus de données
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    push_to_hub=False,
    remove_unused_columns=False,
    label_names=["labels"],
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

## Étape 7 : Entraînement !

In [ ]:
# C'est parti ! (prend environ 3-6h sur Colab T4 gratuit)
trainer.train()

## Étape 8 : Sauvegarder le modèle

In [ ]:
# Sauvegarder localement
trainer.save_model("./whisper-wolof-final")
processor.save_pretrained("./whisper-wolof-final")

# Sauvegarder sur Google Drive
!cp -r ./whisper-wolof-final /content/drive/MyDrive/whisper-wolof-final

print("Modèle sauvegardé !")

In [ ]:
# OU Publier sur Hugging Face Hub (optionnel)
# trainer.push_to_hub("whisper-wolof-finetuned")
# processor.push_to_hub("whisper-wolof-finetuned")

## Étape 9 : Tester le modèle fine-tuné

In [ ]:
from transformers import pipeline

# Charger le modèle fine-tuné
pipe = pipeline(
    "automatic-speech-recognition",
    model="./whisper-wolof-final",
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device="cuda",
)

# Tester avec un audio
result = pipe("/content/drive/MyDrive/wolof_data/audios/test.wav")
print(f"Transcription: {result['text']}")

## Utiliser le modèle fine-tuné dans Wolof Transcriber

Une fois le modèle entraîné, téléchargez-le depuis Google Drive et placez-le dans :
```
wolof-transcriber/backend/models/whisper-wolof-final/
```

Puis changez dans `app.py` :
```python
# Remplacez:
model = whisper.load_model(MODEL_SIZE)

# Par:
from transformers import pipeline
model = pipeline("automatic-speech-recognition", model="./models/whisper-wolof-final")
```